In [ ]:
#@title Installation -- Run this cell to install AiZynthFinder

# Downgrade pip to avoid metadata issues mentioned in warnings
!pip install --upgrade pip==23.3.2

# Uninstall existing torchvision to ensure a clean install
!pip uninstall -y torchvision

# Install compatible torchvision version using PyTorch's official wheel index for torch 2.2.0 and CUDA 12.1
# torchvision 0.17.0 is compatible with torch 2.2.0
!pip install --quiet torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cu121

# Install aizynthfinder
!pip install --quiet aizynthfinder[all]

# Remove the explicit Pillow install as it caused conflicts; let aizynthfinder handle it
# !pip install --ignore-installed Pillow==9.0.0

!mkdir --parents data && download_public_data data

In [ ]:
from aizynthfinder.aizynthfinder import AiZynthFinder

In [ ]:
filename = "data/config.yml"
finder = AiZynthFinder(configfile=filename)

In [ ]:
finder.stock.select("zinc")
finder.expansion_policy.select("uspto")
finder.filter_policy.select("uspto")

In [ ]:
finder.target_smiles = "Cc1cccc(c1N(CC(=O)Nc2ccc(cc2)c3ncon3)C(=O)C4CCS(=O)(=O)CC4)C"
finder.tree_search()

In [ ]:
finder.build_routes()
stats = finder.extract_statistics()

In [ ]:
target_smiles_for_expansion = "Cc1cccc(c1N(CC(=O)Nc2ccc(cc2)c3ncon3)C(=O)C4CCS(=O)(=O)CC4)C"

# After tree_search() and build_routes(), the reactions are available within the 'finder.routes'
# We will extract all unique reaction SMILES from the found routes.
all_route_reactions = set()
if finder.routes: # Check if routes exist
    for route in finder.routes:
        # Each 'route' is a dictionary, and the reaction tree is under the 'reaction_tree' key
        if 'reaction_tree' in route and route['reaction_tree']:
            # Access the 'reactions' attribute of the ReactionTree object by calling it
            for reaction_obj in route['reaction_tree'].reactions():
                # Each reaction_obj should have a 'smiles' attribute
                if hasattr(reaction_obj, 'smiles'):
                    all_route_reactions.add(reaction_obj.smiles)

reactions = list(all_route_reactions)
print(f"Found {len(reactions)} unique reactions from the routes.")
for i, reaction_smiles in enumerate(reactions):
    print(f"Reaction {i+1}: {reaction_smiles}")

In [ ]:
reactants_smiles = []
for reaction_smiles_string in reactions:
    # Assuming reaction_smiles_string is in the format 'reactants>>products'
    if '>>' in reaction_smiles_string:
        reactants_part = reaction_smiles_string.split('>>')[0]
        individual_reactants = reactants_part.split('.')
        reactants_smiles.append(individual_reactants)
    else:
        # If for some reason there's no '>>', treat the whole string as reactants
        individual_reactants = reaction_smiles_string.split('.')
        reactants_smiles.append(individual_reactants)

print("Extracted Reactants for each reaction:")
for i, reactant_list in enumerate(reactants_smiles):
    print(f"Reaction {i+1} Reactants: {', '.join(reactant_list)}")

In [ ]:
import pandas as pd
metadata = []
for reaction_smiles_string in reactions:
    # Assuming reaction_smiles_string itself is a piece of metadata we want to collect
    metadata.append({'reaction_smiles': reaction_smiles_string})
df = pd.DataFrame(metadata)

In [ ]:
from aizynthfinder.chem import Molecule
help(Molecule)
help(Molecule.fingerprint)

In [ ]:
print(df)